# KATHE 2026 — Kashmiri Machine Translation Pipeline

This notebook runs the complete **KATHE 2026** English → Kashmiri translation pipeline on GPU (Google Colab / Kaggle GPU).

### Pipeline Steps:
1. **Environment Setup** (Dependencies & Hugging Face authentication)
2. **Target Script Verification** (`sample_submission.csv` check for `kas_Arab` vs `kas_Deva`)
3. **GPU Smoke Test** (20 rows on GPU with `rerank.py`)
4. **Single Model Rerank Run** (1B model + Reverse 1B reranker)
5. **Winning Ensemble Run** (1B + 200M ensemble + Reverse 1B reranker)
6. **Pre-flight Validation** (`validate_submission.py` checks format, row count, script leakage)
7. **(Optional) LoRA Fine-tuning** (BPCC en-kas stretch goal)

## 1. Setup & Environment

In [ ]:
# Install PyTorch CUDA and dependencies
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

import os
from getpass import getpass

# Prompt for Hugging Face Token (accept model terms at https://huggingface.co/ai4bharat/indictrans2-en-indic-1B first)
if "HF_TOKEN" not in os.environ:
    hf_token = getpass("Enter your Hugging Face Access Token: ")
    os.environ["HF_TOKEN"] = hf_token

## 2. Check Target Script (`kas_Arab` vs `kas_Deva`)
Upload `sample_submission.csv` from Kaggle Data tab to verify the expected target script.

In [ ]:
!python check_script.py sample_submission.csv

## 3. GPU Smoke Test (20 rows)

In [ ]:
!python rerank.py --limit 20 --output /tmp/smoke.csv --nbest 4 --fp16
!head -n 5 /tmp/smoke.csv

## 4. Single-Model Reranking Run (~25–40 min on GPU)

In [ ]:
!python rerank.py \
  --forward ai4bharat/indictrans2-en-indic-1B \
  --nbest 8 --num-beams 8 --alpha 0.5 --tgt-lang kas_Arab \
  --output submission.csv --fp16

## 5. Winning Setup: 2-Model Ensemble + Round-Trip Reranking (~1-1.5 h)

In [ ]:
!python rerank.py \
  --forward ai4bharat/indictrans2-en-indic-1B ai4bharat/indictrans2-en-indic-dist-200M \
  --nbest 8 --num-beams 10 --alpha 0.5 --length-penalty 1.0 \
  --output submission_ens.csv --fp16

## 6. Pre-flight Validation & Kaggle Submission

In [ ]:
!python validate_submission.py submission_ens.csv data/englishdev.csv
# Optional: Submit via Kaggle API if kaggle.json is configured:
# !kaggle competitions submit -c kathe-2026 -f submission_ens.csv -m "1B+200M ensemble, round-trip rerank"

## 7. (Optional) LoRA Fine-Tuning

In [ ]:
# Step A: Train LoRA adapter
!python finetune.py --max-samples 50000 --epochs 1 --output-dir out/lora-kas

# Step B: Run reranking with fine-tuned LoRA weights
!python rerank.py --lora out/lora-kas --output submission_lora.csv --fp16